# Assignment C: Decision Tree vs Random Forest using California Housing

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

housing=fetch_california_housing(as_frame=True)
X=housing.data
y=housing.target

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.3,random_state=42
)


## Part A - Decision Tree Regressor

In [ ]:

depths=[1,2,3,5,8,12,None]
train_mse=[]
test_mse=[]

for d in depths:
    model=DecisionTreeRegressor(max_depth=d,random_state=42)
    model.fit(X_train,y_train)
    train_mse.append(mean_squared_error(y_train,model.predict(X_train)))
    test_mse.append(mean_squared_error(y_test,model.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot([str(d) for d in depths],train_mse,marker='o',label='Train MSE')
plt.plot([str(d) for d in depths],test_mse,marker='s',label='Test MSE')
plt.xlabel("Max Depth")
plt.ylabel("Mean Squared Error")
plt.title("Decision Tree Performance")
plt.legend()
plt.grid(True)
plt.show()



**Observation**

- Small depth (1–3): High bias and underfitting.
- Medium depth (5–12): Better balance.
- Fully grown tree: Near-zero training error but larger testing error, indicating overfitting (high variance).


## Part B - Random Forest Regressor

In [ ]:

estimators=[1,5,10,20,50,100,200]
rf_train=[]
rf_test=[]

for n in estimators:
    rf=RandomForestRegressor(n_estimators=n,random_state=42,n_jobs=-1)
    rf.fit(X_train,y_train)
    rf_train.append(mean_squared_error(y_train,rf.predict(X_train)))
    rf_test.append(mean_squared_error(y_test,rf.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(estimators,rf_train,marker='o',label='Train MSE')
plt.plot(estimators,rf_test,marker='s',label='Test MSE')
plt.xlabel("Number of Trees")
plt.ylabel("Mean Squared Error")
plt.title("Random Forest Performance")
plt.legend()
plt.grid(True)
plt.show()



**Observation**

As the number of trees increases, test MSE decreases and then plateaus because averaging many trees reduces variance. Train MSE stays almost constant since each tree already fits the training data well.


## Part C - Bias–Variance Study

In [ ]:

from sklearn.utils import resample

feature="MedInc"
grid=np.linspace(X[feature].min(),X[feature].max(),300)
X_plot=np.tile(X.mean().values,(300,1))
idx=list(X.columns).index(feature)
X_plot[:,idx]=grid

plt.figure(figsize=(9,5))
for i in range(30):
    Xb,yb=resample(X_train,y_train,random_state=i)
    dt=DecisionTreeRegressor(random_state=i)
    dt.fit(Xb,yb)
    plt.plot(grid,dt.predict(X_plot),alpha=0.2)

plt.xlabel(feature)
plt.ylabel("Predicted House Value")
plt.title("30 Bootstrapped Decision Trees")
plt.show()

rf=RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)
rf.fit(X_train,y_train)

plt.figure(figsize=(9,5))
plt.plot(grid,rf.predict(X_plot),linewidth=2)
plt.xlabel(feature)
plt.ylabel("Predicted House Value")
plt.title("Random Forest Prediction")
plt.show()



### Conclusion

Decision Trees exhibit low bias but high variance when fully grown. Random Forest uses bootstrap aggregation (bagging) to train multiple trees on different samples and averages their predictions. Averaging reduces variance without significantly increasing bias, resulting in better generalization and lower testing MSE. After around 100–200 trees, improvements become marginal because the ensemble has already stabilized.
